In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from pipeline_helpers import ARTIFACT_DIR, DATA_PATH, SEED

print(f"Data: {DATA_PATH}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Reproducibility seed: {SEED}")
from pipeline_helpers import phase7
result = phase7()
display(result["artifact_files"])

Data: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\data\raw\europe_cereal_yield_climate_1990_2022.csv
Artifacts: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline
Reproducibility seed: 42


,file,path
0,best_yield_model.joblib,D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline\best_yield_model.joblib
1,feature_manifest.json,D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline\feature_manifest.json
2,heldout_predictions.csv,D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline\heldout_predictions.csv
3,predicted_vs_actual.png,D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline\predicted_vs_actual.png
4,residuals_vs_predicted.png,D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline\residuals_vs_predicted.png


## Model input contract

In [2]:
display(pd.DataFrame({"required_input_column_in_order": result["feature_manifest"]["required_input_columns_in_order"]}))
print("Export metadata:")
print(result["feature_manifest"])
print(f"Model file size: {result['model_size_bytes']:,} bytes")

Export metadata:
{'target': 'YIELD', 'model_file': 'best_yield_model.joblib', 'selected_candidate': 'RandomForest|context=True', 'model_type': 'RandomForest', 'best_hyperparameters': {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}, 'numeric_feature_columns': ['WB_CCKP_CDD', 'WB_CCKP_CSDI', 'WB_CCKP_CWD', 'WB_CCKP_HURS', 'WB_CCKP_PR', 'WB_CCKP_R50MM', 'WB_CCKP_RX5DAY', 'WB_CCKP_TNN', 'WB_CCKP_TR', 'WB_CCKP_TR29', 'WB_CCKP_TR32', 'WB_CCKP_TX84RR', 'WB_CCKP_TXX', 'WB_CCKP_WSDI', 'YEAR'], 'categorical_feature_columns': ['AREA', 'ITEM'], 'required_input_columns_in_order': ['WB_CCKP_CDD', 'WB_CCKP_CSDI', 'WB_CCKP_CWD', 'WB_CCKP_HURS', 'WB_CCKP_PR', 'WB_CCKP_R50MM', 'WB_CCKP_RX5DAY', 'WB_CCKP_TNN', 'WB_CCKP_TR', 'WB_CCKP_TR29', 'WB_CCKP_TR32', 'WB_CCKP_TX84RR', 'WB_CCKP_TXX', 'WB_CCKP_WSDI', 'YEAR', 'AREA', 'ITEM'], 'context_features_included': True, 'dropped_leakage_columns': ['AREA_HARVESTED', 'PRODUCTION_QUANTITY'], 'climate_reduction': {'source_climate

,required_input_column_in_order
0,WB_CCKP_CDD
1,WB_CCKP_CSDI
2,WB_CCKP_CWD
3,WB_CCKP_HURS
4,WB_CCKP_PR
5,WB_CCKP_R50MM
6,WB_CCKP_RX5DAY
7,WB_CCKP_TNN
8,WB_CCKP_TR
9,WB_CCKP_TR29


## Model input contract

The exported model is refit on all available cleaned rows after held-out evaluation, using the CV-selected hyperparameters. The reported test metrics still come from the untouched holdout predictions saved in Phase 6.

In [3]:
import joblib, json
model = joblib.load(ARTIFACT_DIR / "best_yield_model.joblib")
with (ARTIFACT_DIR / "feature_manifest.json").open(encoding="utf-8") as f:
    manifest = json.load(f)
source = pd.read_pickle(ARTIFACT_DIR / "phase2_no_leakage.pkl")
example = source.loc[[0], manifest["required_input_columns_in_order"]]
print("Example input:")
display(example)
print("Example prediction:", float(model.predict(example)[0]))

Example input:
Example prediction: 1546.2480000000007


,WB_CCKP_CDD,WB_CCKP_CSDI,WB_CCKP_CWD,WB_CCKP_HURS,WB_CCKP_PR,WB_CCKP_R50MM,WB_CCKP_RX5DAY,WB_CCKP_TNN,WB_CCKP_TR,WB_CCKP_TR29,WB_CCKP_TR32,WB_CCKP_TX84RR,WB_CCKP_TXX,WB_CCKP_WSDI,YEAR,AREA,ITEM
0,27.05,5.6,14.53,70.44,963.9,0.4,99.14,-8.07,66.02,0.02,0.0,0.05,24.78,21.31,1990,Albania,Barley


Saved deliverables include the fitted model, its ordered input-column contract, held-out prediction CSV, and diagnostic figures. The model does not require AREA_HARVESTED or PRODUCTION_QUANTITY.